In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

---

# Módulo 4: De ElGamal a ECC

**Dónde estamos:** Podemos sumar puntos, multiplicarlos por escalares y comprimir
claves públicas. Antes de saltar a las firmas (Módulo 5), necesitamos entender
el esquema del que nació la ECC — porque la ECC no inventó una idea nueva,
trasplantó una existente a un grupo más difícil.

Esa historia comienza con un problema: ¿cómo acuerdan dos personas un secreto
compartido por un canal público?

Podrías preguntar: ¿por qué necesitamos un secreto *compartido* si tenemos claves
públicas? ¿No puede Bob simplemente cifrar directamente con la clave pública de
Alice? Sí puede — y el cifrado ElGamal (sección 4.2) hace exactamente eso. Pero
internamente, "cifrar con una clave pública" *es* establecer un secreto compartido.
Cuando Bob elige un $k$ aleatorio y calcula $A^k$ usando la clave pública de Alice,
está creando un secreto compartido de un solo uso que solo Alice puede reconstruir
(usando su clave privada sobre el $C_1 = g^k$ efímero de Bob). Diffie-Hellman es
la primitiva que hace esto posible. Entenderlo primero hace que todo lo demás —
cifrado ElGamal, ECDH, incluso firmas — sea solo variaciones del mismo truco.

## 4.1 Intercambio de Claves Diffie-Hellman (1976)

Imagina que Alice y Bob quieren comunicarse de forma segura, pero todos pueden ver
sus mensajes. No pueden simplemente enviar una contraseña — un espía la leería.

La idea de Diffie y Hellman: usar una función unidireccional para que ambas partes
puedan llegar independientemente al **mismo secreto**, sin nunca transmitirlo.

**Configuración:** Todos acuerdan un primo grande $p$ y un generador $g$ — un número
cuyos poderes $g^1, g^2, g^3, \ldots \bmod p$ recorren todos los valores de $1$
a $p-1$.

**El intercambio:**

```
           Alice                          Bob
           ─────                          ───
  1. Elige secreto a                1. Elige secreto b
  2. Calcula A = g^a mod p          2. Calcula B = g^b mod p
  3. Envía A a Bob ──────────────►  3. Recibe A
  4. Recibe B ◄──────────────────── 4. Envía B a Alice
  5. Calcula B^a = g^(ba) mod p     5. Calcula A^b = g^(ab) mod p
           │                              │
           └──── mismo secreto compartido ┘
                    g^(ab) mod p
```

**Por qué funciona:** Alice calcula $B^a = (g^b)^a = g^{ab}$. Bob calcula
$A^b = (g^a)^b = g^{ab}$. Misma respuesta — conmutatividad de exponentes.

**Por qué es seguro:** Nadie puede recuperar la clave privada de otro — ni
siquiera los participantes. Alice conoce $a$ y ve $B = g^b$, pero no puede
encontrar $b$. Bob conoce $b$ y ve $A = g^a$, pero no puede encontrar $a$.
Y el espía ve tanto $A$ como $B$ pero no puede encontrar $a$ o $b$, ni puede
calcular $g^{ab}$ sin uno de ellos. Extraer un exponente de $g^a$ es el problema
del logaritmo discreto — inviable para $p$ grandes.

**Ejemplo concreto con números pequeños ($p = 23$, $g = 5$):**

| | Alice | Bob |
|---|---|---|
| Secreto | $a = 6$ | $b = 15$ |
| Público | $A = 5^6 \bmod 23 = 8$ | $B = 5^{15} \bmod 23 = 19$ |
| Secreto compartido | $19^6 \bmod 23 = 2$ | $8^{15} \bmod 23 = 2$ |

El espía ve 8 y 19 pero no puede descifrar que el secreto compartido es 2
sin conocer 6 o 15.

## 4.2 ElGamal — del intercambio de claves al cifrado

En 1985, Taher ElGamal extendió Diffie-Hellman a un esquema de cifrado completo.

En DH, Alice y Bob participan ambos — cada uno contribuye un secreto y llegan a un
resultado compartido. Pero ¿qué pasa si Bob quiere enviar un mensaje a Alice y ella
no está en línea para hacer un intercambio? ElGamal resuelve esto: Bob usa la
**clave pública publicada** de Alice para crear un secreto compartido de un solo
uso *por sí mismo*, luego lo usa para cifrar.

Piénsalo como **Diffie-Hellman unilateral**: Bob hace el intercambio solo, usando
la clave pública de Alice como su mitad.

**Configuración:** Igual que DH — primo $p$, generador $g$. Alice publica su clave
pública $A = g^a \bmod p$ (donde $a$ es su secreto).

### Cifrado (Bob → Alice)

```
  Bob quiere enviar mensaje m a Alice.
  Él conoce: p, g, y la clave pública de Alice A = g^a

  Paso 1: Elige un k aleatorio (fresco, de un solo uso — como un nonce)
  Paso 2: Calcula C₁ = g^k mod p
  Paso 3: Calcula el secreto compartido S = A^k mod p
  Paso 4: Cifra el mensaje: C₂ = m · S mod p
  Paso 5: Envía (C₁, C₂) a Alice.
```

### Descifrado (Alice)

```
  Alice recibe (C₁, C₂). Conoce su secreto a.

  Paso 1: Recalcula el secreto compartido S = C₁^a mod p
  Paso 2: Descifra: m = C₂ · S⁻¹ mod p
```

### El problema con ElGamal sobre enteros

El esquema funciona, pero el logaritmo discreto en enteros mod $p$ puede ser
atacado con **cálculo de índices** — un algoritmo subexponencial. Para estar
seguro, necesitas claves enormes: 3072 bits para seguridad de 128 bits.

## 4.3 ECC — mismo esquema, grupo más difícil

La observación de Koblitz y Miller (1985): **ejecutar los mismos esquemas, pero
reemplazar exponenciación mod $p$ con multiplicación escalar en una curva elíptica.**

Todo se mapea directamente:

| | ElGamal (enteros mod $p$) | ECC (curva elíptica) |
|---|---|---|
| Parámetro público | primo $p$, generador $g$ | curva, punto generador $G$ |
| Clave privada | exponente secreto $a$ | escalar secreto $d$ |
| Clave pública | $A = g^a \bmod p$ | $P = d \times G$ |
| Problema difícil | dado $A$, encontrar $a$ | dado $P$, encontrar $d$ |
| Intercambio de claves | $g^{ab}$ | $d_A \cdot d_B \cdot G$ |
| Mejor ataque | subexponencial (cálculo de índices) | exponencial (rho de Pollard) |

La diferencia crítica: no hay ataque subexponencial conocido contra el logaritmo
discreto de curvas elípticas. El mejor ataque (rho de Pollard) es $O(\sqrt{N})$,
que es **completamente exponencial**. Misma seguridad, una fracción del tamaño
de clave.

El código a continuación demuestra ambos esquemas lado a lado — ElGamal con
enteros, luego lo mismo con puntos de curvas elípticas.

In [ ]:
# Lado a lado: intercambio de claves ElGamal vs ECC

print("=" * 60)
print("Intercambio de Claves ElGamal (enteros mod p)")
print("=" * 60)

# Ejemplo pequeño de ElGamal (del ensayo)
p_eg = 101
alpha = 7  # Raíz primitiva mod 101

# Alice
a_priv = 23  # Clave privada de Alice
beta_a = pow(alpha, a_priv, p_eg)  # Clave pública de Alice
print(f"Alice: privada a={a_priv}, pública β = α^a mod p = 7^{a_priv} mod 101 = {beta_a}")

# Bob
b_priv = 37  # Clave privada de Bob
beta_b = pow(alpha, b_priv, p_eg)  # Clave pública de Bob
print(f"Bob:   privada b={b_priv}, pública β'= α^b mod p = 7^{b_priv} mod 101 = {beta_b}")

# Secreto compartido
shared_eg_a = pow(beta_b, a_priv, p_eg)  # Alice calcula β'^a
shared_eg_b = pow(beta_a, b_priv, p_eg)  # Bob calcula β^b
print(f"Secreto compartido: Alice={shared_eg_a}, Bob={shared_eg_b}, Coinciden={shared_eg_a == shared_eg_b}")
print(f"(Ambos calculan α^(ab) mod p = 7^{a_priv*b_priv} mod 101 = {pow(alpha, a_priv*b_priv, p_eg)})")

print(f"\n{'=' * 60}")
print("Intercambio de Claves ECC (ECDH en secp256k1)")
print("=" * 60)

# Alice
alice_priv = secrets.randbelow(SECP_N - 1) + 1
alice_pub = scalar_mult(alice_priv, G)  # Clave pública de Alice = a × G
print(f"Alice: privada a (256 bits aleatorios), pública A = a×G")

# Bob
bob_priv = secrets.randbelow(SECP_N - 1) + 1
bob_pub = scalar_mult(bob_priv, G)  # Clave pública de Bob = b × G
print(f"Bob:   privada b (256 bits aleatorios), pública B = b×G")

# Secreto compartido: ambos calculan el mismo punto
shared_ecc_a = scalar_mult(alice_priv, bob_pub)   # a × (b×G) = ab×G
shared_ecc_b = scalar_mult(bob_priv, alice_pub)    # b × (a×G) = ab×G
print(f"Punto del secreto compartido: {shared_ecc_a == shared_ecc_b}  ✓")
print(f"Ambos calculan a×b×G (mismo punto, nunca transmitido)")

print(f"\n{'─' * 60}")
print(f"ElGamal: seguridad de α^a mod p  (necesita p de ~3072 bits)")
print(f"ECC:     seguridad de k×G         (necesita N de ~256 bits)")
print(f"Misma seguridad, claves 12× más pequeñas.")

## 4.2 Cifrado / Descifrado ECC (ElGamal en Curvas)

El ensayo demuestra el cifrado ECC usando la curva $y^2 = x^3 - x + 4$ sobre $\mathbb{F}_{457}$.
El esquema se mapea directamente desde ElGamal:

| ElGamal | Análogo ECC |
|---------|------------|
| $\beta = \alpha^a \bmod p$ | $Q = d \times G$ (clave pública) |
| Cifrar: $(\alpha^k, m \cdot \beta^k)$ | Cifrar: $(k \times G, \; P_m + k \times Q)$ |
| Descifrar: $m = t \cdot (\beta')^{-a}$ | Descifrar: $P_m = C_2 - d \times C_1$ |

La multiplicación en ElGamal se convierte en **suma de puntos** en ECC.
La exponenciación se convierte en **multiplicación escalar**.

In [ ]:
from ecc.small_curve import (
    SmallPoint, small_add, small_mult, small_negate, small_sqrt,
    encode_char_to_point, decode_point_to_char,
    P_SMALL, A_SMALL, B_SMALL, G_SMALL, K_ENC,
)

assert (8**2) % P_SMALL == (4**3 + A_SMALL*4 + B_SMALL) % P_SMALL, "¡G no está en la curva!"

d = 101
Q = small_mult(d, G_SMALL)
print(f"=== Cifrado ECC (Ejemplo del Ensayo) ===")
print(f"Curva: y² = x³ - x + 4 sobre F_457")
print(f"G = {G_SMALL}")
print(f"Clave privada d = {d}")
print(f"Clave pública Q = d×G = {Q}")

msg_val = 7
pm = encode_char_to_point(msg_val)
print(f"\nMensaje: 'H' (valor {msg_val}) → punto {pm}")

k_rand = 41
C1 = small_mult(k_rand, G_SMALL)
C2 = small_add(pm, small_mult(k_rand, Q))
print(f"\nCifrar con k aleatorio={k_rand}:")
print(f"  C1 = k×G = {C1}")
print(f"  C2 = Pm + k×Q = {C2}")

dC1 = small_mult(d, C1)
pm_recovered = small_add(C2, small_negate(dC1))
msg_recovered = decode_point_to_char(pm_recovered)
print(f"\nDescifrar:")
print(f"  d×C1 = {dC1}")
print(f"  Pm = C2 - d×C1 = {pm_recovered}")
print(f"  Decodificado: valor {msg_recovered} → '{chr(65 + msg_recovered)}'")
print(f"  Coincide: {pm == pm_recovered}  ✓")

### Por qué funciona el descifrado

$$C_2 - d \times C_1 = (P_m + k \times Q) - d \times (k \times G)$$
$$= P_m + k \times (d \times G) - d \times (k \times G)$$
$$= P_m + k \cdot d \times G - d \cdot k \times G$$
$$= P_m \quad \checkmark$$

El factor aleatorio $k$ se cancela porque ambas partes tienen acceso al
secreto compartido $k \cdot d \times G$ a través de diferentes caminos.

---